# Microsam Cell Probabilities

See if microsam can predict foreground vs. background in DIC hyphal images.

The model is able to resolve cell from background, but due to the watershed post-processing, it is unable to resolve overlapping hyphal cells. Also, the foreground vs. background predictions need some finetuning.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = (
    "~/A8/Data_Wang/260213_HWPA_DAPI/",
    )

search_path = (
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/03",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/04",
    "~/A8/Data_Roan/260327_Cocultures_Mutants_Phalloidin/CET155/05",
)

file_pattern = ("CROP_*.tif",
                "MAX_*",
                "SHARPEST*C1*.tif",
                "*DIC.tif",
                "MAX_CET145*")

config = {
    "preprocessing": {
        "resize_image": True,
        "max_dim": 1024,
        "outlier_percentile": 0.35,
        "quantization": "8bit",
        "correct_DIC_shift" : [0,0] # [Y,X] more negative Y to shift up, more positive X to shift to right
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
dic_files = []
for p in search_path:
    dic_files.extend(find_files_by_pattern(p, file_pattern[3], verbose=True))

dic_imgs = [ImageContainer(img,config) for img in dic_files]

In [ ]:
from micro_sam import util
import torch
from micro_sam.instance_segmentation import AMGBase, InstanceSegmentationWithDecoder, get_decoder, get_instance_segmentation_generator
from typing import Optional, Union, Tuple, Dict, Any
import os
import numpy as np

def get_predictor_and_segmenter(
    model_type: str,
    checkpoint_path: Union[os.PathLike, str],
    decoder_path: Union[os.PathLike, str],
    device: str = None,
    amg: Optional[bool] = None,
    is_tiled: bool = False,
    **kwargs,
) -> Tuple[util.SamPredictor, Union[AMGBase, InstanceSegmentationWithDecoder]]:
    """Get the Segment Anything model and class for automatic instance segmentation.

    Args:
        model_type: The Segment Anything model choice.
        checkpoint_path: The filepath to the stored model checkpoints.
        decoder_path: The filepath to the stored decoder checkpoints.
        device: The torch device. By default, automatically chooses the best available device.
        amg: Whether to perform automatic segmentation in AMG mode.
            Otherwise AIS will be used, which requires a special segmentation decoder.
            If not specified AIS will be used if it is available and otherwise AMG will be used.
        is_tiled: Whether to return segmenter for performing segmentation in tiling window style.
            By default, set to 'False'.
        kwargs: Keyword arguments for the automatic mask generation class.

    Returns:
        The Segment Anything model.
        The automatic instance segmentation class.
    """
    if device is None:
        device = util.get_device(device=device)

    predictor, state = util.get_sam_model(
        model_type=model_type, device=device, checkpoint_path=checkpoint_path, return_state=True,
    )
    state["decoder_state"] = torch.load(decoder_path, map_location=device, weights_only=False)

    if amg is None:
        amg = "decoder_state" not in state

    if amg:
        decoder = None
    else:
        if "decoder_state" not in state:
            raise RuntimeError("You have passed 'amg=False', but your model does not contain a segmentation decoder.")
        decoder_state = state["decoder_state"]
        decoder = get_decoder(image_encoder=predictor.model.image_encoder, decoder_state=decoder_state, device=device)

    segmenter = get_instance_segmentation_generator(predictor=predictor, is_tiled=is_tiled, decoder=decoder, **kwargs)

    return predictor, segmenter

def run_automatic_instance_segmentation(
    image: np.ndarray,
    ndim: int,
    checkpoint_path: Union[os.PathLike, str],
    decoder_path: Union[os.PathLike, str],
    model_type: str = "vit_l_lm",
    device: Optional[Union[str, torch.device]] = None,
    tile_shape: Optional[Tuple[int, int]] = None,
    halo: Optional[Tuple[int, int]] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, util.ImageEmbeddings]:
    """Automatic Instance Segmentation (AIS) by training an additional instance decoder in SAM.

    NOTE: AIS is supported only for `µsam` models.

    Args:
        image: The input image.
        ndim: The number of dimensions for the input data.
        checkpoint_path: The path to stored SAM checkpoints.
        decoder_path: The path to stored decoder checkpoints.
        model_type: The choice of the `µsam` model.
        device: The device to run the model inference.
        tile_shape: The tile shape for tiling-based segmentation.
        halo: The overlap shape on each side per tile for stitching the segmented tiles.

    Returns:
        instances: The instance segmentation mask.
        foreground: The raw foreground probability map from the decoder, shape (H, W).
        center_distances: The center distance predictions from the decoder, shape (H, W).
        boundary_distances: The boundary distance predictions from the decoder, shape (H, W).
        image_embeddings: The image embeddings. A zarr Group keyed by tile index when tiling
            is used, or a dict with a (1, 256, 64, 64) numpy array otherwise.
    """
    predictor, segmenter = get_predictor_and_segmenter(
        model_type=model_type,
        checkpoint_path=checkpoint_path,
        decoder_path=decoder_path,
        device=device,
        amg=False,
        is_tiled=(tile_shape is not None),
    )

    image_embeddings = util.precompute_image_embeddings(
        predictor=predictor,
        input_=image,
        ndim=ndim,
        tile_shape=tile_shape,
        halo=halo,
    )

    initialize_kwargs = dict(image=image, image_embeddings=image_embeddings)
    generate_kwargs = {"output_mode": None}

    if tile_shape is not None:
        initialize_kwargs["batch_size"] = 1
        generate_kwargs.update({"tile_shape": tile_shape, "halo": halo})

    segmenter.initialize(**initialize_kwargs)
    instances = segmenter.generate(**generate_kwargs)

    decoder_state = segmenter.get_state()

    return (
        instances,
        decoder_state["foreground"],
        decoder_state["center_distances"],
        decoder_state["boundary_distances"],
        image_embeddings,
    )


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

def save_decoder_visualization(
    container,
    instances: np.ndarray,
    foreground: np.ndarray,
    center_distances: np.ndarray,
    boundary_distances: np.ndarray,
    output_path: Union[str, Path],
    title: str = "",
) -> None:
    """Save a 2x3 figure of the original image, AIS segmentation, and the three decoder maps.

    Args:
        container: ImageContainer for the source image; its display() method is used.
        instances: AIS instance segmentation mask, shape (H, W), integer-labeled.
        foreground: Foreground probability map from the decoder, shape (H, W).
        center_distances: Center distance predictions from the decoder, shape (H, W).
        boundary_distances: Boundary distance predictions from the decoder, shape (H, W).
        output_path: Filepath to save the figure (e.g. a .png path).
        title: Optional super-title for the figure.
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    container.display(title="Original Image", ax=axes[0, 0])

    # Instance segmentation: mask background (label 0) to black, color instances distinctly.
    masked_instances = np.ma.masked_equal(instances, 0)
    cmap_instances = plt.cm.nipy_spectral.copy()
    cmap_instances.set_bad(color="black")
    axes[0, 1].imshow(instances == 0, cmap="gray_r", interpolation="nearest")  # black background
    axes[0, 1].imshow(masked_instances, cmap=cmap_instances, interpolation="nearest")
    axes[0, 1].set_title("AIS Segmentation")
    axes[0, 1].axis("off")

    for ax, data, label in [
        (axes[0, 2], foreground,         "Foreground"),
        (axes[1, 0], center_distances,   "Center Distances"),
        (axes[1, 1], boundary_distances, "Boundary Distances"),
    ]:
        im = ax.imshow(data, cmap="viridis", vmin=0, vmax=1)
        ax.set_title(label)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    axes[1, 2].axis("off")

    if title:
        fig.suptitle(title, fontsize=13)

    fig.tight_layout()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


In [ ]:
from micro_sam.util import get_device

sam_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm").expanduser()
decoder_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm_decoder").expanduser()
model_type = 'vit_l_lm'
device = get_device()

predictor, segmenter = get_predictor_and_segmenter(checkpoint_path=sam_checkpoint,
                                                   decoder_path=decoder_checkpoint,
                                                   device=device,
                                                   model_type=model_type,
                                                   amg=False,
                                                   is_tiled=False)

In [ ]:
segmenter._decoder

In [ ]:
from micro_sam.util import get_device

# Model directories
sam_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm").expanduser()
decoder_checkpoint = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm_decoder").expanduser()
model_type = 'vit_l_lm'
device = get_device()

# Save outputs
all_processed_images = {}
all_decoder_outputs = {}  # stores instances + the three decoder maps per image

# Compute Loop
for container in dic_imgs:
    file_name = container.name
    print(f"Processing {file_name}...")

    processed_image = container.merge()

    if processed_image is None:
        print(f"  Skipping {file_name} due to preprocessing error.")
        continue

    all_processed_images[file_name] = processed_image

    instances, foreground, center_distances, boundary_distances, image_embeddings = run_automatic_instance_segmentation(
        image=processed_image,
        ndim=2,
        checkpoint_path=str(sam_checkpoint),
        decoder_path=str(decoder_checkpoint),
        model_type=model_type,
        device=device,
        # tile_shape=(512,512),
        # halo=(64,64),
    )

    if instances is not None:
        all_decoder_outputs[file_name] = {
            "instances": instances,
            "foreground": foreground,
            "center_distances": center_distances,
            "boundary_distances": boundary_distances,
        }
        print(f"  Generated automatic instance segmentation for {file_name}.")

# Visualization loop
base_input_dir = Path("~/A8/Data_Wang/").expanduser()
base_output_dir = Path("output/microSAM_AIS_foreground")
print(f"\nDecoder visualizations will be saved to: {base_output_dir.resolve()}")

for container in dic_imgs:
    file_name = container.name

    if file_name not in all_decoder_outputs:
        continue

    file_path = container._source_paths[0]
    try:
        relative_dir = file_path.parent.relative_to(base_input_dir)
    except ValueError:
        relative_dir = file_path.parent.name

    current_output_dir = base_output_dir / relative_dir
    output_filename = current_output_dir / f"{file_path.stem}_decoder.png"

    print(f"Creating visualization for {file_name}...")
    decoder = all_decoder_outputs[file_name]
    save_decoder_visualization(
        container=container,
        instances=decoder["instances"],
        foreground=decoder["foreground"],
        center_distances=decoder["center_distances"],
        boundary_distances=decoder["boundary_distances"],
        output_path=output_filename,
        title=file_name,
    )
    print(f"  -> Saved to {output_filename}")

print("\nAll decoder visualizations saved.")


In [ ]:
output_filename